# Loss functions

> A set of custom loss functions

In [2]:
#| default_exp losses

In [3]:
#| hide
from nbdev.showdoc import *

# Loss Functions

In [4]:
#| export
from fastai.vision.all import *
from monai.losses import SSIMLoss


In [5]:
#| export
class CombinedLoss:
    "losses combined"
    def __init__(self, spatial_dims=2, alpha=0.33, beta=0.33):
        store_attr()
        self.SSIM_loss = SSIMLoss(spatial_dims=spatial_dims)
        self.MSE_loss =  nn.MSELoss()
        self.MAE_loss =  nn.L1Loss()
        
    def __call__(self, pred, targ):
        return (1 - self.alpha - self.beta) * self.SSIM_loss(pred, targ) + self.alpha * self.MSE_loss(pred, targ) + self.beta * self.MAE_loss(pred, targ)

In [6]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Make sure the inputs are probabilities
        inputs = torch.sigmoid(inputs)

        # Flatten tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        # Calculate the intersection
        intersection = (inputs * targets).sum()

        # Compute Dice Coefficient
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)

        # Copmute dice loss
        loss = 1 - dice

        return loss


# inputs and targets must be equally dimensional tensors

inputs = torch.randn((1, 1, 256, 256))  # Input
targets = torch.randint(0, 2, (1, 1, 256, 256)).float()  # Ground Truth


# Inicialize
dice_loss = DiceLoss()

# Compute loss
loss = dice_loss(inputs, targets)
print('Dice Loss:', loss.item())

print(inputs)
print(targets)

Dice Loss: 0.4994472861289978
tensor([[[[ 1.0045,  0.2688, -1.4387,  ...,  0.2748, -1.1894, -0.5855],
          [-0.5506,  0.6496,  1.4599,  ...,  0.2549,  0.1396, -0.5395],
          [ 0.8324, -0.8579,  0.2426,  ...,  0.8287,  1.9277, -0.6439],
          ...,
          [-1.1500, -2.3167,  1.7069,  ...,  0.9731,  0.6509, -0.1228],
          [-0.2695, -0.2966, -0.3618,  ...,  0.7237,  1.1687,  0.8964],
          [ 0.5460,  1.5734, -0.1824,  ...,  0.0159, -0.7043, -0.3882]]]])
tensor([[[[0., 0., 0.,  ..., 1., 0., 1.],
          [1., 0., 0.,  ..., 1., 1., 0.],
          [1., 0., 1.,  ..., 0., 1., 1.],
          ...,
          [1., 0., 0.,  ..., 1., 0., 1.],
          [0., 1., 1.,  ..., 1., 1., 0.],
          [1., 0., 1.,  ..., 1., 0., 1.]]]])


# Metrics

In [7]:
#| export

def SSIM(x, y, spatial_dims=2):
    return 1 - SSIMLoss(spatial_dims)(x,y)

SSIMMetric = AvgMetric(SSIM)

In [8]:
#| hide
import nbdev; nbdev.nbdev_export()